# Exercise 3: Apertium pipeline trace

*Machine Translation · version 0.1.0*

> Install Apertium and the Kazakh–Tatar language pair (or, without an Apertium installation, run its stages by hand with the Kazakh analyser, the pair's bilingual dictionary and the Tatar generator). Translate the Kazakh sentence *Бүгін ауа райы өте жақсы.* ("The weather is very nice today.") and trace each stage of the pipeline: morphological analysis output, lexical transfer output, structural transfer output, and morphological generation output. What does the trace reveal about how each stage handles this sentence?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" torch transformers sentencepiece sacrebleu requests` (the compiled `hfst` package needs Python 3.11 or 3.12; elsewhere install `pyhfst`). NLLB-200 distilled 600M is loaded from `NLLB_PATH` (a local copy) or from the Hugging Face Hub at the revision pinned in the book; it runs in float32 on the CPU, about 1.7 s per sentence on four cores. Translations are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so an interrupted run resumes where it stopped. Test data come from the commit-pinned TIL release (X-WMT test sets, MIT licence); the X-WMT files are tokenised, and the code removes the spaces around punctuation before scoring (a heuristic, see `detok`). Scores use SacreBLEU with its default settings: BLEU with the `13a` tokeniser, chrF, and chrF++ (`word_order=2`).

**What runs here.** The notebook runs the stages without the Apertium engine. Analysis and generation use the Apertium Kazakh analyser and Tatar generator that TurkicNLP downloads; lexical transfer uses the bilingual dictionary of the released `apertium-kaz-tat` pair (commit `028c57b`), read in the Kazakh→Tatar direction. Two stages are replaced: instead of the pair's Constraint Grammar disambiguation, the first reading whose lemma and part of speech are in the dictionary is kept, and the structural-transfer rules (`.t1x`/`.t2x`) are not executed. With Apertium installed, `echo "Бүгін ауа райы өте жақсы." | apertium -d . kaz-tat` and the `kaz-tat-morph`, `kaz-tat-disam`, `kaz-tat-biltrans` and `kaz-tat-transfer` modes of the pair print the real intermediate stages.

In [ ]:
import collections, hashlib, os, re, statistics, time
from pathlib import Path
import requests

# Commit-pinned TIL release files (X-WMT test sets and the corpus README); MIT licence.
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
FETCHED = {}                                   # url -> sha256 of every file read (saved with the outputs)

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def detok(s):
    """Undo the X-WMT tokenisation (spaces around punctuation) so references match raw MT output."""
    s = re.sub(r" ([’'] ?)", lambda m: m.group(1).strip(), s)       # Sialkot ’ taki -> Sialkot’taki
    s = re.sub(r" ([.,!?;:%)\]»])", r"\1", s)
    return re.sub(r"([(\[«]) ", r"\1", s).strip()

def xwmt(src, tgt, n=None):
    """The first n sentence pairs of the X-WMT test set for src-tgt (X-WMT language codes)."""
    s = fetch(f"{TIL}xwmt/test/{src}-{tgt}/{src}-{tgt}.{src}").read_text(encoding="utf-8").splitlines()
    t = fetch(f"{TIL}xwmt/test/{src}-{tgt}/{src}-{tgt}.{tgt}").read_text(encoding="utf-8").splitlines()
    assert len(s) == len(t), (src, tgt)
    pairs = [(detok(a), detok(b)) for a, b in zip(s, t) if a.strip() and b.strip()]
    return pairs[:n] if n else pairs

NLLB_CODE = {"az": "azj_Latn", "ba": "bak_Cyrl", "kk": "kaz_Cyrl", "ky": "kir_Cyrl", "tr": "tur_Latn",
             "uz": "uzn_Latn", "en": "eng_Latn", "ru": "rus_Cyrl"}
NLLB = {}

def nllb():
    """NLLB-200 distilled 600M: from NLLB_PATH (a local copy) or the pinned Hub revision; float32, CPU."""
    if not NLLB:
        import torch
        from transformers import AutoModelForSeq2SeqLM, NllbTokenizer
        path = os.environ.get("NLLB_PATH", "facebook/nllb-200-distilled-600M")
        rev = None if Path(path).exists() else "f8d333a098d19b4fd9a8b18f94170487ad3f821d"
        NLLB["tok"] = NllbTokenizer.from_pretrained(path, revision=rev)
        NLLB["model"] = AutoModelForSeq2SeqLM.from_pretrained(path, revision=rev, torch_dtype=torch.float32).eval()
        NLLB["torch"] = torch
        torch.set_num_threads(os.cpu_count() or 1)
    return NLLB["tok"], NLLB["model"]

CACHE_FILE = CACHE / "nllb-translations.jsonl"       # translations are cached, so a run can resume
LAST_SECONDS = 0.0

def _cached():
    import json
    out = {}
    if CACHE_FILE.exists():
        for line in CACHE_FILE.read_text(encoding="utf-8").splitlines():
            r = json.loads(line); out[tuple(r["key"])] = (r["out"], r["capped"], r["sec"])
    return out

def translate(sents, src, tgt, beams=1, batch=8, max_new=256):
    """Translate with explicit source and target language tokens (greedy unless beams > 1).
    Returns (outputs, number of outputs that reached the generation ceiling without an
    end-of-sequence token). Results are cached per sentence in CACHE_FILE, with the generation
    time of their batch divided by its size; LAST_SECONDS holds the total for the last call."""
    global LAST_SECONDS
    import json
    cache = _cached()
    key = lambda s: [src, tgt, beams, max_new, s]
    todo = [s for s in dict.fromkeys(sents) if tuple(key(s)) not in cache]
    if todo:
        tok, model = nllb()
        tok.src_lang = NLLB_CODE[src]
        CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
        with NLLB["torch"].inference_mode(), CACHE_FILE.open("a", encoding="utf-8") as f:
            for i in range(0, len(todo), batch):
                chunk = todo[i:i + batch]
                t0 = time.time()
                enc = tok(chunk, return_tensors="pt", padding=True)
                gen = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(NLLB_CODE[tgt]),
                                     num_beams=beams, do_sample=False, max_new_tokens=max_new)
                sec = (time.time() - t0) / len(chunk)
                for s, row, text in zip(chunk, gen, tok.batch_decode(gen, skip_special_tokens=True)):
                    capped = int(row[-1] != tok.eos_token_id and row[-1] != tok.pad_token_id)
                    cache[tuple(key(s))] = (text, capped, sec)
                    f.write(json.dumps({"key": key(s), "out": text, "capped": capped, "sec": sec}, ensure_ascii=False) + "\n")
                f.flush()
    res = [cache[tuple(key(s))] for s in sents]
    LAST_SECONDS = sum(r[2] for r in res)
    return [r[0] for r in res], sum(r[1] for r in res)

def metrics(hyps, refs):
    """Corpus BLEU (13a), chrF and chrF++ with SacreBLEU, rounded, plus the chrF++ signature."""
    from sacrebleu.metrics import BLEU, CHRF
    bleu, chrf, chrfpp = BLEU(tokenize="13a"), CHRF(), CHRF(word_order=2)
    r = {"BLEU": bleu.corpus_score(hyps, [refs]).score, "chrF": chrf.corpus_score(hyps, [refs]).score,
         "chrF++": chrfpp.corpus_score(hyps, [refs]).score}
    return {k: round(v, 2) for k, v in r.items()}, str(chrfpp.get_signature())

In [ ]:
try:
    import hfst
except ImportError:
    import sys, pyhfst
    sys.modules["hfst"] = pyhfst
import xml.etree.ElementTree as ET
import turkicnlp

# Apertium resources: the Kazakh analyser and Tatar generator shipped with TurkicNLP, and the
# kaz-tat bilingual dictionary of the released pair (apertium-kaz-tat, commit 028c57b, GPL).
for lang in ("kaz", "tat"):
    turkicnlp.download(lang, processors=["morph"])
models = Path(os.environ.get("TURKICNLP_MODELS_DIR", "~/.turkicnlp/models")).expanduser()
analyser = hfst.HfstInputStream(str(models / "kaz/Cyrl/morph/apertium/kaz.automorf.hfst")).read()
generator = hfst.HfstInputStream(str(models / "tat/Cyrl/morph/apertium/tat.autogen.hfst")).read()
BIDIX = ("https://raw.githubusercontent.com/apertium/apertium-kaz-tat/"
         "028c57beeaf8c5434a3af126b36047eb8d27630a/apertium-kaz-tat.kaz-tat.dix")

def side(el):
    text = el.text or ""
    for c in el:
        text += f"<{c.get('n')}>" if c.tag == "s" else " " if c.tag == "b" else ""
        text += c.tail or ""
    return text

bidix = collections.defaultdict(list)                 # 'lemma<pos>' -> Tatar 'lemma<pos>' (kaz->tat entries)
for e in ET.parse(fetch(BIDIX)).getroot().iter("e"):
    p = e.find("p")
    if p is not None and e.get("r") != "RL":
        bidix[side(p.find("l"))].append(side(p.find("r")))
print(f"bilingual dictionary: {len(bidix)} Kazakh entries usable in the kaz->tat direction")

sentence = "Бүгін ауа райы өте жақсы."
words = re.findall(r"\w+|[^\w\s]", sentence)
print(f"\n{'word':8} | 1. analyses | 2. kept | 3. lexical transfer | 5. generation")
target = []
for w in words:
    analyses = [r[0] for r in analyser.lookup(w)] or [r[0] for r in analyser.lookup(w.lower())]
    analyses = [a for a in analyses if "+" not in a]           # drop copula continuations
    # 2. selection: keep readings whose lemma+POS is in the bilingual dictionary (a stand-in for
    #    Apertium's Constraint Grammar disambiguation, which this notebook does not run)
    kept = [a for a in analyses if re.match(r"[^<]+<[^>]+>", a).group(0) in bidix] or analyses
    a = kept[0]
    key = re.match(r"[^<]+<[^>]+>", a).group(0)
    tgt_lemma = bidix.get(key, [key])[0]
    tgt_analysis = tgt_lemma + a[len(key):]
    forms = [r[0] for r in generator.lookup(tgt_analysis)] or ["#" + tgt_lemma.split("<")[0]]
    target.append(forms[0])
    print(f"{w:8} | {', '.join(analyses[:4])}{' …' if len(analyses) > 4 else ''} | {a} | {tgt_analysis} | {forms[0]}")
print("\nTatar output (4. structural transfer not run):", " ".join(target).replace(" .", "."))

## Worked solution

Output of `experiments/ch10_exercises/run_ch10_exercises.py`:

```
bilingual dictionary: 9204 Kazakh entries usable in the kaz->tat direction

word     | 1. analyses | 2. kept | 3. lexical transfer | 5. generation
Бүгін    | бүгін<adv>, бүгін<n><attr>, бүгін<n><nom> | бүгін<adv> | бүген<adv> | бүген
ауа      | ау<v><iv><gna_impf>, ау<v><iv><prc_impf>, ауа<n><attr>, ауа<n><nom> | ауа<n><attr> | һава<n><attr> | һава
райы     | рай<n><px3sp><nom> | рай<n><px3sp><nom> | торыш<n><px3sp><nom> | торышы
өте      | өт<v><iv><gna_impf>, өт<v><iv><prc_impf>, өт<v><tv><gna_impf>, өт<v><tv><prc_impf> … | өте<adv> | бик<adv> | бик
жақсы    | жақсы<adj>, жақсы<adj><advl>, жақсы<adj><subst><nom>, жақсы<adv> | жақсы<adj> | яхшы<adj> | яхшы
.        | .<sent> | .<sent> | .<sent> | .

Tatar output (4. structural transfer not run): бүген һава торышы бик яхшы.
```

**Analysis.** The analyser returns several readings for most words: *ауа* is a noun 'air' but also two verb forms of *ау-* 'to move, to shift'; *өте* is the adverb 'very' and four verb forms of *өт-* 'to pass'; *бүгін* can be an adverb or a noun. Capitalised *Бүгін* is analysed only after lower-casing. Choosing among these readings is the job of the disambiguation stage; a wrong choice here (*өте* as a verb) would be carried through every later stage.

**Selection and lexical transfer.** Keeping readings that the dictionary can translate picks *бүгін* as an adverb, *ауа* as a noun and *өте* as an adverb, which is right in this sentence, but only because the verb readings' lemmas have different dictionary entries. The dictionary maps *бүгін* → *бүген*, *ауа* → *һава*, *өте* → *бик* and *жақсы* → *яхшы*. The interesting entry is *рай* → *торыш*: Kazakh *ауа райы* 'weather' (literally 'state of the air') becomes Tatar *һава торышы*, a different noun in the same possessive construction. The tags (`<n><px3sp><nom>` for the third-person possessive) are passed through unchanged.

**Structural transfer.** Not run here. For this sentence nothing needs reordering: both languages are SOV with the same adjective and adverb positions, so the chunk-level rules would leave the order as it is. A sentence with a relative clause or a construction that Tatar expresses differently would need them.

**Generation.** The Tatar generator produces *бүген һава торышы бик яхшы.* from the transferred analyses, which is correct Tatar for 'The weather is very nice today.' The capital letter of the first word is lost, because this trace does not carry the surface case of the input through; the Apertium pipeline keeps it and restores it on output.

## Your turn — extension

1. Install Apertium (`apertium-kaz-tat`) and compare the real stage outputs with this trace, especially the disambiguation of *өте*.
2. Trace *Мен кеше кітапханаға бардым.* 'I went to the library yesterday.' Which stage handles the dative and the first-person past?